# Notebook 04 — Ground Truth and Pairs

Tasks:
- build record map
- build positive pairs
- build random negatives
- build hard negatives
- verify labels
- measure class balance
- check leakage

In [ ]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

OUTPUT_DIR = "D:/data viewer/data/processed"

## Load Data

In [ ]:
master = pd.read_csv(f"{OUTPUT_DIR}/master_projects.csv")
sources = {
    "crm": pd.read_csv(f"{OUTPUT_DIR}/crm_export_corrupted.csv"),
    "erp": pd.read_csv(f"{OUTPUT_DIR}/erp_export_corrupted.csv"),
    "partner": pd.read_excel(f"{OUTPUT_DIR}/partner_export_corrupted.xlsx"),
    "document": pd.read_csv(f"{OUTPUT_DIR}/document_metadata_corrupted.csv"),
}

print(f"Master: {len(master)} projects")
for name, df in sources.items():
    print(f"  {name}: {len(df)} records")

## Build Record Map

In [ ]:
id_cols = {"crm": "project_id", "erp": "project_reference", "partner": "external_reference", "document": "project_reference"}

records = []
for source_name, df in sources.items():
    id_col = id_cols[source_name]
    for _, row in df.iterrows():
        records.append({"source_name": source_name, "source_record_id": str(row[id_col]), "master_project_id": str(row[id_col])})

record_map = pd.DataFrame(records)
record_map.to_csv(f"{OUTPUT_DIR}/record_map.csv", index=False)
print(f"Record map: {len(record_map)} rows")
record_map.head()

## Build Positive Pairs

In [ ]:
pairs = []
pair_id = 0
source_names = list(sources.keys())

for _, proj in master.iterrows():
    pid = proj["project_id"]
    records_for_project = []
    for source_name in source_names:
        df = sources[source_name]
        id_col = id_cols[source_name]
        matching = df[df[id_col] == pid]
        if len(matching) > 0:
            records_for_project.append((source_name, str(matching.iloc[0][id_col])))
    for i in range(len(records_for_project)):
        for j in range(i + 1, len(records_for_project)):
            pair_id += 1
            pairs.append({"pair_id": f"PAIR-{pair_id:06d}", "source_a": records_for_project[i][0], "source_b": records_for_project[j][0], "record_a_id": records_for_project[i][1], "record_b_id": records_for_project[j][1], "master_id_a": pid, "master_id_b": pid, "label": 1})

pos_pairs = pd.DataFrame(pairs)
print(f"Positive pairs: {len(pos_pairs)}")
pos_pairs.head()

## Build Negative Pairs

In [ ]:
neg_pairs = []
pair_id = 100000
master_ids = master["project_id"].tolist()

for _ in range(2000):
    pid_a, pid_b = random.sample(master_ids, 2)
    src_a = random.choice(source_names)
    src_b = random.choice([s for s in source_names if s != src_a])
    df_a = sources[src_a]
    df_b = sources[src_b]
    id_col_a = id_cols[src_a]
    id_col_b = id_cols[src_b]
    rec_a = df_a[df_a[id_col_a] == pid_a]
    rec_b = df_b[df_b[id_col_b] == pid_b]
    if len(rec_a) == 0 or len(rec_b) == 0:
        continue
    pair_id += 1
    neg_pairs.append({"pair_id": f"PAIR-{pair_id:06d}", "source_a": src_a, "source_b": src_b, "record_a_id": str(rec_a.iloc[0][id_col_a]), "record_b_id": str(rec_b.iloc[0][id_col_b]), "master_id_a": pid_a, "master_id_b": pid_b, "label": 0})

neg_pairs = pd.DataFrame(neg_pairs)
print(f"Negative pairs: {len(neg_pairs)}")
neg_pairs.head()

## Build Hard Negatives

In [ ]:
hard_neg = []
pair_id = 200000

for _ in range(500):
    pid_a, pid_b = random.sample(master_ids, 2)
    proj_a = master[master["project_id"] == pid_a].iloc[0]
    proj_b = master[master["project_id"] == pid_b].iloc[0]
    if (proj_a["customer_name"] == proj_b["customer_name"] or proj_a["installer"] == proj_b["installer"] or proj_a["address"] == proj_b["address"]):
        src_a = random.choice(["crm", "erp", "partner"])
        src_b = random.choice([s for s in ["crm", "erp", "partner"] if s != src_a])
        df_a = sources[src_a]
        df_b = sources[src_b]
        id_col_a = id_cols[src_a]
        id_col_b = id_cols[src_b]
        rec_a = df_a[df_a[id_col_a] == pid_a]
        rec_b = df_b[df_b[id_col_b] == pid_b]
        if len(rec_a) > 0 and len(rec_b) > 0:
            pair_id += 1
            hard_neg.append({"pair_id": f"PAIR-{pair_id:06d}", "source_a": src_a, "source_b": src_b, "record_a_id": str(rec_a.iloc[0][id_col_a]), "record_b_id": str(rec_b.iloc[0][id_col_b]), "master_id_a": pid_a, "master_id_b": pid_b, "label": 0})

hard_neg = pd.DataFrame(hard_neg)
print(f"Hard negative pairs: {len(hard_neg)}")
hard_neg.head()

## Combine and Verify

In [ ]:
all_pairs = pd.concat([pos_pairs, neg_pairs, hard_neg], ignore_index=True)
all_pairs.to_csv(f"{OUTPUT_DIR}/entity_pairs.csv", index=False)

print(f"Total pairs: {len(all_pairs)}")
print(f"\nClass balance:")
print(all_pairs["label"].value_counts())

print(f"\nLeakage check:")
pos_check = all_pairs[all_pairs["label"] == 1]
neg_check = all_pairs[all_pairs["label"] == 0]
print(f"  Positives with different master_id: {(pos_check['master_id_a'] != pos_check['master_id_b']).sum()} (should be 0)")
print(f"  Negatives with same master_id: {(neg_check['master_id_a'] == neg_check['master_id_b']).sum()} (should be 0)")